# 08. Molecular Dynamics Trajectories & Radial Distribution Functions

Molecular dynamics (MD) tracks how atoms move over time, and two summary metrics appear in nearly every simulation study:
- **RMSD (Root-Mean-Square Deviation)** measures how far atoms drift from a starting reference frame after optimal alignment.
- **Radial Distribution Function ($g(r)$)** measures how likely you are to find a neighbor atom at distance $r$ relative to a uniform random gas. Sharp peaks mark well-defined first- and second-neighbor **coordination (or solvation) shells**, while $g(r) \to 1$ at long range.

MD post-processing has two classic traps: atoms wrapping across periodic box walls create fake RMSD spikes, and naive RDF histograms lose accuracy past half a skewed box length. `xkcd-matchem` guards against both:
- `Trajectory.rmsd()` unwraps cumulative periodic boundary jumps between consecutive frames before optional Kabsch superposition, and supports atom subsets via `indices=`.
- `Trajectory.compute_rdf()` integrates **exact spherical-shell volumes** $\frac{4\pi}{3}(r_{\text{out}}^3 - r_{\text{in}}^3)$ over all periodic image shells enclosing `r_max`—and offers `strict=True` to verify that `r_max` stays within the **Wigner-Seitz minimum-image inradius** (the largest sphere fitting inside the box without encountering self-images).

In [ ]:
# Install xkcd-matchem automatically in Google Colab
import importlib.util


if importlib.util.find_spec("xkcd_matchem") is None:
    %pip install -q "git+https://github.com/vzordillo/xkcd-matchem.git"


## Basic Usage

Let's load a bundled 10-frame LAMMPS trajectory (`cu_md.lammpstrj`) to sketch its RMSD trace alongside a precomputed GROMACS liquid-water O–O radial distribution curve (`water_rdf.xvg`):

In [ ]:
import matplotlib.pyplot as plt

import xkcd_matchem as xm


md = xm.load_trajectory("cu_md.lammpstrj")
rdf_xvg = xm.load("water_rdf.xvg")

with xm.xkcd_style("classic"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.2, 4.4))
    xm.plot_md_timeseries(
        md, quantity="rmsd", running_window=3, ax=ax1, title="Cu MD RMSD Trace"
    )
    xm.plot_rdf(
        rdf_xvg.x,
        rdf_xvg.y,
        unit="nm",
        ax=ax2,
        title="Liquid Water O-O RDF (XVG)",
    )
    fig.tight_layout()

## Advanced Usage

To compute a periodic radial distribution function directly from trajectory frames while preventing periodic self-image artifacts, first check the safe Wigner-Seitz inradius with `max_min_image_radius()`, then call `md.compute_rdf(..., strict=True)`. Below we also evaluate RMSD over a 4-atom subset:

In [ ]:
r_ws = xm.core.geometry.max_min_image_radius(md[0].cell, md[0].pbc)
r_safe = round(0.95 * r_ws, 2)
print(
    f"Wigner-Seitz inradius: {r_ws:.2f} A -> strict g(r) up to {r_safe} A"
)

r_vals, g_vals = md.compute_rdf(
    ("Cu", "Cu"), r_max=r_safe, n_bins=30, strict=True
)
sub_rmsd = md.rmsd(ref_index=0, indices=[0, 1, 2, 3])
print("4-atom subset RMSD max (A):", round(float(sub_rmsd.max()), 4))

ax = xm.plot_rdf(
    r_vals,
    g_vals,
    unit="A",
    color="#9C755F",
    title="Computed Cu-Cu Periodic RDF (strict=True)",
)

:::{tip}
**Try Next:** For LAMMPS trajectories that store numeric atom types (`1`, `2`, ...) instead of an `element` column, pass `type_map={1: "Cu", 2: "O"}` to `xm.io.native.lammps.read_lammpstrj("dump.lammpstrj", type_map=...)`. Want to export `md` as a looping animated GIF? See [Tutorial 15: Animated GIFs](15_animations_md_orbits_and_isosurfaces.ipynb). Continue to **Tutorial 09** for phase hulls, EOS fits, and ML parity benchmarks.
:::